# 01. DART 데이터 수집

이 노트북은 `02_dart_data_collection.md` 지침에 맞춰 OpenDART API에서 사업보고서를 수집합니다.

분석 단위는 회사명이 아니라 `stock_code × fiscal_year`입니다. `company_master.csv`의 381개 기업-연도에서 시작해, 직전 회계연도 사업보고서를 KCGS 평가연도와 연결합니다.

생성 산출물은 두 종류입니다.

- `data/processed/dart_reports/{stock_code}_{fiscal_year}.json`: 기업-연도별 원문 섹션과 ESG passage 후보
- `data/processed/dart_passages.csv`: 후속 전처리와 분석에 바로 쓰는 passage 단위 표
- `data/processed/dart_collection_status.csv`: 381행 전체 수집 시도 결과와 실패 사유

## 1. 설정, 경로, API 키 확인

`.env`, `DART_API_KEY`, `OPENDART_API_KEY` 순서로 API 키를 읽습니다. 노트북 실행 위치가 프로젝트 루트가 아니어도 `data/` 또는 `pyproject.toml`을 기준으로 루트를 찾습니다.

In [14]:
import csv
import io
import json
import os
import re
import time
import zipfile
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from html import unescape
from pathlib import Path
from typing import Any

import pandas as pd
import requests


def find_project_root() -> Path:
    """현재 작업 위치에서 프로젝트 루트를 찾습니다."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').exists() or (candidate / 'data').exists():
            return candidate
    return Path.cwd()


def load_env_file(env_path: Path) -> None:
    """간단한 KEY=VALUE 형식의 .env 파일을 환경변수로 읽습니다."""
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


PROJECT_ROOT = find_project_root()
load_env_file(PROJECT_ROOT / '.env')

API_KEY = os.getenv('DART_API_KEY') or os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('OpenDART API 키가 없습니다. .env에 DART_API_KEY 또는 OPENDART_API_KEY를 설정하세요.')

OPENDART_BASE = 'https://opendart.fss.or.kr/api'
REQUEST_TIMEOUT = 60
SLEEP_BETWEEN_CALLS = 0.30

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROCESSED_DIR / 'dart_reports'

COMPANY_MASTER_PATH = RAW_DIR / 'company_master.csv'
SEED_DICTIONARY_PATH = RAW_DIR / 'seed_dictionary.csv'
CORP_CODE_PATH = RAW_DIR / 'CORPCODE.xml'
PASSAGES_CSV_PATH = PROCESSED_DIR / 'dart_passages.csv'
STATUS_CSV_PATH = PROCESSED_DIR / 'dart_collection_status.csv'

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'프로젝트 루트: {PROJECT_ROOT}')
print(f'회사 목록: {COMPANY_MASTER_PATH}')
print(f'출력 폴더: {REPORT_DIR}')
print(f'API Key 앞 8자리: {API_KEY[:8]}...')

프로젝트 루트: /Users/home1/Documents/esg-dart-project
회사 목록: /Users/home1/Documents/esg-dart-project/data/raw/company_master.csv
출력 폴더: /Users/home1/Documents/esg-dart-project/data/processed/dart_reports
API Key 앞 8자리: 42a1b845...


## 2. 입력 데이터 점검

`company_master.csv`는 381행이어야 하며, `stock_code × fiscal_year`가 고유해야 합니다. `stock_code`는 숫자로 읽히면 앞의 0이 사라지므로 항상 6자리 문자열로 보정합니다.

In [15]:
def load_company_master(path: Path = COMPANY_MASTER_PATH) -> pd.DataFrame:
    """company_master.csv를 읽고 분석 키를 정규화합니다."""
    df = pd.read_csv(
        path,
        dtype={
            'stock_code': 'string',
            'corp_code': 'string',
            'rcept_no': 'string',
        },
    )
    df = df.copy()
    df['stock_code'] = df['stock_code'].str.replace(r'\.0$', '', regex=True).str.zfill(6)
    df['fiscal_year'] = df['fiscal_year'].astype(int)
    df['esg_year'] = df['esg_year'].astype(int)
    df['report_fiscal_year'] = df['fiscal_year']
    return df


def validate_company_master(df: pd.DataFrame) -> dict[str, Any]:
    """수집 전 입력 표의 기본 조건을 확인합니다."""
    issues = []
    if len(df) != 381:
        issues.append(f'행 수가 381이 아닙니다: {len(df)}')
    if df['stock_code'].nunique() != 127:
        issues.append(f'고유 stock_code 수가 127이 아닙니다: {df["stock_code"].nunique()}')
    duplicate_count = int(df.duplicated(['stock_code', 'fiscal_year']).sum())
    if duplicate_count:
        issues.append(f'중복 stock_code × fiscal_year 행이 있습니다: {duplicate_count}')
    mismatch = df[df['esg_year'] != df['fiscal_year'] + 1]
    if len(mismatch):
        issues.append(f'esg_year != fiscal_year + 1 행이 있습니다: {len(mismatch)}')
    missing_year_sets = (
        df.groupby('stock_code')['fiscal_year']
        .apply(lambda years: sorted(set(years)) != [2022, 2023, 2024])
    )
    bad_stock_count = int(missing_year_sets.sum())
    if bad_stock_count:
        issues.append(f'2022, 2023, 2024 세 연도를 모두 갖지 않는 stock_code가 있습니다: {bad_stock_count}')
    return {
        'rows': len(df),
        'unique_stock_code': int(df['stock_code'].nunique()),
        'fiscal_years': sorted(df['fiscal_year'].unique().tolist()),
        'esg_years': sorted(df['esg_year'].unique().tolist()),
        'duplicate_stock_year_rows': duplicate_count,
        'issues': issues,
    }


company_master = load_company_master()
input_report = validate_company_master(company_master)

print(json.dumps(input_report, ensure_ascii=False, indent=2))
company_master.head(10)

{
  "rows": 381,
  "unique_stock_code": 127,
  "fiscal_years": [
    2022,
    2023,
    2024
  ],
  "esg_years": [
    2023,
    2024,
    2025
  ],
  "duplicate_stock_year_rows": 0,
  "issues": []
}


,company_name,corp_code,stock_code,industry,fiscal_year,report_code,rcept_no,esg_source,esg_grade,e_grade,s_grade,g_grade,esg_year,report_fiscal_year
0,삼성전자,<NA>,005930,전기전자,2022,11011,<NA>,한국ESG기준원,A,A,A+,B+,2023,2022
1,삼성전자,<NA>,005930,전기전자,2023,11011,<NA>,한국ESG기준원,B+,B+,A,B,2024,2023
2,삼성전자,<NA>,005930,전기전자,2024,11011,<NA>,한국ESG기준원,A,B+,A+,B+,2025,2024
3,BYC,<NA>,001460,섬유/의류,2022,11011,<NA>,한국ESG기준원,D,D,D,D,2023,2022
4,BYC,<NA>,001460,섬유/의류,2023,11011,<NA>,한국ESG기준원,D,D,D,C,2024,2023
5,BYC,<NA>,001460,섬유/의류,2024,11011,<NA>,한국ESG기준원,D,D,D,C,2025,2024
6,현대자동차,<NA>,005380,자동차,2022,11011,<NA>,한국ESG기준원,A,A,A,B+,2023,2022
7,현대자동차,<NA>,005380,자동차,2023,11011,<NA>,한국ESG기준원,A,A+,B+,A,2024,2023
8,현대자동차,<NA>,005380,자동차,2024,11011,<NA>,한국ESG기준원,A,A+,B+,B+,2025,2024
9,CJ대한통운,<NA>,000120,물류,2022,11011,<NA>,한국ESG기준원,A,A,A+,B+,2023,2022


## 3. `corp_code` 매핑

OpenDART `corpCode.xml`은 전체 회사 코드 ZIP을 반환합니다. 한 번 내려받아 `data/raw/CORPCODE.xml`로 캐시한 뒤, `stock_code`에서 `corp_code`를 찾습니다.

In [16]:
def download_corp_code_xml(force: bool = False) -> Path:
    """OpenDART corpCode.xml을 다운로드하고 캐시합니다."""
    if CORP_CODE_PATH.exists() and not force:
        print(f'캐시 사용: {CORP_CODE_PATH}')
        return CORP_CODE_PATH

    url = f'{OPENDART_BASE}/corpCode.xml'
    response = requests.get(url, params={'crtfc_key': API_KEY}, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()

    if response.content[:2] != b'PK':
        snippet = response.content[:500].decode('utf-8', errors='replace')
        raise RuntimeError(f'corpCode.xml 응답이 ZIP이 아닙니다: {snippet}')

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
        zip_file.extract('CORPCODE.xml', path=CORP_CODE_PATH.parent)

    print(f'저장 완료: {CORP_CODE_PATH} ({CORP_CODE_PATH.stat().st_size:,} bytes)')
    return CORP_CODE_PATH


def build_stock_to_corp_map() -> dict[str, str]:
    """CORPCODE.xml에서 {stock_code: corp_code} 매핑을 만듭니다."""
    if not CORP_CODE_PATH.exists():
        download_corp_code_xml()

    tree = ET.parse(CORP_CODE_PATH)
    root = tree.getroot()
    candidates: dict[str, dict[str, str]] = {}

    for item in root.findall('list'):
        stock_code = (item.findtext('stock_code') or '').strip().zfill(6)
        corp_code = (item.findtext('corp_code') or '').strip()
        modify_date = (item.findtext('modify_date') or '').strip()
        if not stock_code.strip('0') or not corp_code:
            continue
        previous = candidates.get(stock_code)
        if previous is None or modify_date > previous['modify_date']:
            candidates[stock_code] = {'corp_code': corp_code, 'modify_date': modify_date}

    return {stock_code: row['corp_code'] for stock_code, row in candidates.items()}


# 최초 실행 시 네트워크 호출이 발생합니다.
download_corp_code_xml()
stock_to_corp = build_stock_to_corp_map()

missing_corp = sorted(set(company_master['stock_code']) - set(stock_to_corp))
print(f'매핑된 상장 stock_code 수: {len(stock_to_corp):,}')
print(f'company_master 중 corp_code 미매핑: {len(missing_corp)}')
print('삼성전자 005930 ->', stock_to_corp.get('005930'))

캐시 사용: /Users/home1/Documents/esg-dart-project/data/raw/CORPCODE.xml
매핑된 상장 stock_code 수: 3,965
company_master 중 corp_code 미매핑: 0
삼성전자 005930 -> 00126380


## 4. 사업보고서 목록 조회와 `rcept_no` 선택

`list.json`에서 `fiscal_year + 1` 공시연도의 사업보고서를 조회합니다. 선택 기준은 대상 회계연도가 보고서명에 표시된 사업보고서이며, `[기재정정]`을 원본보다 우선하고 `[첨부정정]`은 후순위로 둡니다.

In [17]:
@dataclass
class ReportSelection:
    report_nm: str
    rcept_no: str
    rcept_dt: str
    corp_code: str
    stock_code: str
    fiscal_year: int

    @property
    def viewer_url(self) -> str:
        return f'https://dart.fss.or.kr/dsaf001/main.do?rcpNo={self.rcept_no}'


def opendart_get_json(path: str, params: dict[str, Any], retries: int = 2) -> dict[str, Any]:
    """OpenDART JSON API를 호출합니다."""
    request_params = {'crtfc_key': API_KEY, **params}
    last_error = None
    for attempt in range(retries + 1):
        try:
            response = requests.get(
                f'{OPENDART_BASE}/{path}',
                params=request_params,
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'OpenDART {path} 호출 실패: {last_error}')


def list_annual_reports(corp_code: str, fiscal_year: int) -> list[dict[str, Any]]:
    """회계연도 다음 해에 공시된 사업보고서 목록을 조회합니다."""
    disclosure_year = fiscal_year + 1
    payload = opendart_get_json(
        'list.json',
        {
            'corp_code': corp_code,
            'bgn_de': f'{disclosure_year}0101',
            'end_de': f'{disclosure_year}1231',
            'pblntf_detail_ty': 'A001',
            'page_count': '100',
        },
    )

    status = payload.get('status')
    if status in {'013', '014'}:
        return []
    if status != '000':
        raise RuntimeError(f'list.json 에러: {status} {payload.get("message")}')
    return payload.get('list', [])


def report_name_has_target_year(report_name: str, fiscal_year: int) -> bool:
    """보고서명에서 대상 회계연도를 확인합니다. 예: 사업보고서 (2024.12)."""
    compact = re.sub(r'\s+', '', report_name)
    return bool(re.search(rf'\({fiscal_year}(?:\.|\))', compact))


def report_priority(row: dict[str, Any], fiscal_year: int) -> tuple[int, int, int]:
    """낮을수록 좋은 보고서 후보입니다."""
    name = row.get('report_nm', '')
    has_target_year = report_name_has_target_year(name, fiscal_year)
    if not has_target_year:
        target_score = 9
    else:
        target_score = 0

    if '[첨부정정]' in name:
        correction_score = 8
    elif '[기재정정]' in name:
        correction_score = 0
    elif name.replace(' ', '').startswith('사업보고서'):
        correction_score = 1
    else:
        correction_score = 4

    rcept_no = int(row.get('rcept_no', '0') or '0')
    return (target_score, correction_score, -rcept_no)


def select_annual_report(corp_code: str, stock_code: str, fiscal_year: int) -> ReportSelection | None:
    """사업보고서 후보 중 지침에 맞는 rcept_no를 선택합니다."""
    reports = list_annual_reports(corp_code, fiscal_year)
    if not reports:
        return None

    sorted_reports = sorted(reports, key=lambda row: report_priority(row, fiscal_year))
    selected = sorted_reports[0]

    if not report_name_has_target_year(selected.get('report_nm', ''), fiscal_year):
        return None

    return ReportSelection(
        report_nm=selected.get('report_nm', ''),
        rcept_no=selected.get('rcept_no', ''),
        rcept_dt=selected.get('rcept_dt', ''),
        corp_code=corp_code,
        stock_code=stock_code,
        fiscal_year=fiscal_year,
    )


# 단건 확인 예시입니다. 네트워크 호출을 줄이려면 필요할 때만 실행하세요.
# samsung = select_annual_report(stock_to_corp['005930'], '005930', 2024)
# samsung

## 5. 보고서 XML 다운로드

`document.xml`은 ZIP 파일을 반환합니다. ZIP 안의 XML 파일을 모두 읽고, 파서가 실패할 수 있는 문서도 고려해 문자열 fallback을 준비합니다.

In [18]:
def fetch_report_xml_files(rcept_no: str) -> list[tuple[str, str]]:
    """document.xml API에서 보고서 ZIP을 받아 XML 텍스트 목록으로 반환합니다."""
    last_error = None
    for attempt in range(3):
        try:
            response = requests.get(
                f'{OPENDART_BASE}/document.xml',
                params={'crtfc_key': API_KEY, 'rcept_no': rcept_no},
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()
            content = response.content
            if content[:2] != b'PK':
                snippet = content[:500].decode('utf-8', errors='replace')
                raise RuntimeError(f'document.xml 에러 응답: {snippet}')

            xml_files = []
            with zipfile.ZipFile(io.BytesIO(content)) as zip_file:
                for name in zip_file.namelist():
                    if not name.lower().endswith('.xml'):
                        continue
                    raw = zip_file.read(name)
                    text = decode_xml_bytes(raw)
                    xml_files.append((name, text))
            return xml_files
        except (requests.RequestException, zipfile.BadZipFile, RuntimeError) as exc:
            last_error = exc
            if attempt < 2:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'document.xml 다운로드 실패: {last_error}')


def decode_xml_bytes(raw: bytes) -> str:
    """DART XML에서 자주 보이는 인코딩을 순서대로 시도합니다."""
    for encoding in ('utf-8', 'euc-kr', 'cp949'):
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue
    return raw.decode('utf-8', errors='replace')

## 6. II, IV, VI 섹션 추출

사업보고서 원문에서 `SECTION-1` 제목을 보고 II, IV, VI 섹션을 추출합니다. 이 세 섹션이 ESG 관련 문단 후보를 찾는 대상입니다.

In [19]:
TARGET_SECTION_LABELS = ('II', 'IV', 'VI')
EXPECTED_SECTION_HEAD = {
    'II': '사업의 내용',
    'IV': '이사의 경영진단',
    'VI': '이사회',
}


def local_name(tag: str) -> str:
    """XML namespace를 제거한 태그명을 반환합니다."""
    return tag.rsplit('}', 1)[-1].upper()


def direct_title_text(element: ET.Element) -> str:
    """SECTION-1 바로 아래 TITLE 텍스트만 읽습니다."""
    for child in list(element):
        if local_name(child.tag) == 'TITLE':
            return ''.join(child.itertext()).strip()
    return ''


def normalize_roman_title(title: str) -> str:
    """로마 숫자 표기를 ASCII 표기로 맞춥니다."""
    return (
        title.replace('Ⅱ', 'II')
        .replace('Ⅳ', 'IV')
        .replace('Ⅵ', 'VI')
        .replace('ⅱ', 'II')
        .replace('ⅳ', 'IV')
        .replace('ⅵ', 'VI')
    )


def classify_section(title: str) -> str | None:
    """섹션 제목이 II, IV, VI 중 하나인지 판별합니다."""
    compact = re.sub(r'\s+', '', normalize_roman_title(title or ''))
    if re.match(r'^II\.?사업의?내용', compact):
        return 'II'
    if re.match(r'^IV\.?이사의?경영진단', compact):
        return 'IV'
    if re.match(r'^VI\.?이사회', compact):
        return 'VI'
    return None


def normalize_text(text: str) -> str:
    """HTML entity와 공백을 분석하기 쉬운 형태로 정리합니다."""
    text = unescape(text)
    text = re.sub(r'[ \t\r\f]+', ' ', text)
    text = re.sub(r'\n[ \t]*', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def element_to_text(element: ET.Element) -> str:
    """XML element를 plain text로 변환합니다."""
    raw = ET.tostring(element, encoding='unicode', method='xml')
    raw = re.sub(r'</(P|TITLE|TR|LI|H[1-6]|TABLE)>', '\n', raw, flags=re.IGNORECASE)
    raw = re.sub(r'<BR\s*/?>', '\n', raw, flags=re.IGNORECASE)
    raw = re.sub(r'<[^>]+>', ' ', raw)
    return normalize_text(raw)


def extract_sections_with_parser(xml_text: str) -> dict[str, str | None]:
    """XML 파서로 SECTION-1 단위 섹션을 추출합니다."""
    result = {label: None for label in TARGET_SECTION_LABELS}
    try:
        root = ET.fromstring(xml_text.encode('utf-8'))
    except ET.ParseError:
        return result

    for element in root.iter():
        if local_name(element.tag) != 'SECTION-1':
            continue
        label = classify_section(direct_title_text(element))
        if label and result[label] is None:
            result[label] = element_to_text(element)
    return result


def extract_sections_with_regex(xml_text: str) -> dict[str, str | None]:
    """파서가 실패한 XML을 SECTION-1 문자열 범위로 보조 추출합니다."""
    result = {label: None for label in TARGET_SECTION_LABELS}
    starts = [match.start() for match in re.finditer(r'<SECTION-1\b[^>]*>', xml_text, flags=re.IGNORECASE)]
    starts.append(len(xml_text))

    for index in range(len(starts) - 1):
        chunk = xml_text[starts[index]:starts[index + 1]]
        title_match = re.search(r'<TITLE\b[^>]*>(.*?)</TITLE>', chunk, flags=re.DOTALL | re.IGNORECASE)
        if title_match is None:
            continue
        title = normalize_text(re.sub(r'<[^>]+>', ' ', title_match.group(1)))
        label = classify_section(title)
        if label and result[label] is None:
            text = re.sub(r'</(P|TITLE|TR|LI|H[1-6]|TABLE)>', '\n', chunk, flags=re.IGNORECASE)
            text = re.sub(r'<BR\s*/?>', '\n', text, flags=re.IGNORECASE)
            text = re.sub(r'<[^>]+>', ' ', text)
            result[label] = normalize_text(text)
    return result


def extract_target_sections(xml_files: list[tuple[str, str]]) -> dict[str, str | None]:
    """ZIP 안의 모든 XML에서 II, IV, VI 섹션을 찾습니다."""
    result = {label: None for label in TARGET_SECTION_LABELS}
    for _, xml_text in xml_files:
        parsed = extract_sections_with_parser(xml_text)
        fallback = extract_sections_with_regex(xml_text) if any(v is None for v in parsed.values()) else {}
        for label in TARGET_SECTION_LABELS:
            if result[label] is None:
                result[label] = parsed.get(label) or fallback.get(label)
    return result

## 7. ESG passage 후보 추출

`seed_dictionary.csv`의 `pattern`을 기준으로 II, IV, VI 섹션에서 ESG 관련 문단 후보를 찾습니다. 이 단계는 완성된 ESG 사전이 아니라, 원문 확인과 후속 전처리를 위한 재현 가능한 후보 추출입니다.

In [20]:
def load_seed_patterns(path: Path = SEED_DICTIONARY_PATH) -> list[dict[str, Any]]:
    """seed_dictionary.csv에서 dimension별 정규식 패턴을 읽습니다."""
    seeds = pd.read_csv(path)
    patterns = []
    for _, row in seeds.iterrows():
        pattern = str(row.get('pattern', '')).strip()
        if not pattern or pattern.lower() == 'nan':
            continue
        patterns.append(
            {
                'dimension': str(row.get('dimension', '')).strip(),
                'seed_term': str(row.get('seed_term', '')).strip(),
                'regex': re.compile(pattern, flags=re.IGNORECASE),
            }
        )
    return patterns


def split_passage_candidates(text: str) -> list[str]:
    """섹션 텍스트를 문단 후보로 나눕니다."""
    if not text:
        return []
    chunks = re.split(r'\n{2,}|(?<=[다음임함음요오])\.\s+', text)
    cleaned = []
    for chunk in chunks:
        passage = normalize_text(chunk)
        if 40 <= len(passage) <= 2000:
            cleaned.append(passage)
    return cleaned


def extract_esg_passages(
    sections: dict[str, str | None],
    seed_patterns: list[dict[str, Any]],
) -> list[dict[str, str]]:
    """II, IV, VI 섹션에서 seed pattern과 매칭되는 passage를 추출합니다."""
    passages = []
    seen = set()

    for section_label, section_text in sections.items():
        for passage in split_passage_candidates(section_text or ''):
            matched_by_dimension: dict[str, set[str]] = {}
            for seed in seed_patterns:
                matches = sorted(set(seed['regex'].findall(passage)))
                if not matches:
                    continue
                matched_by_dimension.setdefault(seed['dimension'], set()).update(str(m) for m in matches)
                matched_by_dimension[seed['dimension']].add(seed['seed_term'])

            for dimension, terms in matched_by_dimension.items():
                key = (section_label, dimension, passage[:250])
                if key in seen:
                    continue
                seen.add(key)
                passages.append(
                    {
                        'section': section_label,
                        'passage': passage,
                        'dimension': dimension,
                        'matched_terms': '|'.join(sorted(term for term in terms if term)),
                    }
                )
    return passages


seed_patterns = load_seed_patterns()
print(f'로드한 seed pattern 수: {len(seed_patterns)}')

로드한 seed pattern 수: 30


## 8. 성공/실패 결과 저장

기업-연도별 JSON은 수집 lineage를 남기고, passage CSV는 후속 분석에 바로 연결합니다. 실패 행은 `dart_collection_status.csv`에 남기며 passage CSV에 가짜 0으로 넣지 않습니다.

In [21]:
PASSAGE_COLUMNS = [
    'company_name',
    'stock_code',
    'corp_code',
    'esg_year',
    'report_fiscal_year',
    'rcept_no',
    'rcept_dt',
    'viewer_url',
    'section',
    'passage',
    'dimension',
    'matched_terms',
]

STATUS_COLUMNS = [
    'row_index',
    'company_name',
    'stock_code',
    'corp_code',
    'fiscal_year',
    'esg_year',
    'report_nm',
    'rcept_no',
    'rcept_dt',
    'viewer_url',
    'status',
    'failure_stage',
    'failure_reason',
    'section_II_chars',
    'section_IV_chars',
    'section_VI_chars',
    'passage_count',
]


def report_json_path(stock_code: str, fiscal_year: int) -> Path:
    return REPORT_DIR / f'{stock_code}_{fiscal_year}.json'


def write_json_atomic(path: Path, payload: dict[str, Any]) -> None:
    """중간 실패로 깨진 JSON이 남지 않도록 임시 파일 후 교체합니다."""
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    tmp_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp_path.replace(path)


def make_empty_status(row_index: int, row: pd.Series) -> dict[str, Any]:
    return {
        'row_index': row_index,
        'company_name': row.get('company_name', ''),
        'stock_code': row.get('stock_code', ''),
        'corp_code': '',
        'fiscal_year': int(row.get('fiscal_year')),
        'esg_year': int(row.get('esg_year')),
        'report_nm': '',
        'rcept_no': '',
        'rcept_dt': '',
        'viewer_url': '',
        'status': 'FAIL',
        'failure_stage': '',
        'failure_reason': '',
        'section_II_chars': 0,
        'section_IV_chars': 0,
        'section_VI_chars': 0,
        'passage_count': 0,
    }


def load_existing_json(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        return None

## 9. 기업-연도 한 행 처리

한 행은 `corp_code` 확인, 사업보고서 선택, 원문 다운로드, 섹션 추출, ESG passage 추출 순서로 처리됩니다. 이미 정상 JSON이 있으면 API 호출을 생략할 수 있습니다.

In [22]:
def process_firm_year(
    row_index: int,
    row: pd.Series,
    skip_existing: bool = True,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    """company_master 한 행을 처리하고 status와 passage rows를 반환합니다."""
    status = make_empty_status(row_index, row)
    stock_code = str(row['stock_code']).zfill(6)
    fiscal_year = int(row['fiscal_year'])
    esg_year = int(row['esg_year'])
    company_name = row.get('company_name', '')
    output_path = report_json_path(stock_code, fiscal_year)

    if skip_existing:
        existing = load_existing_json(output_path)
        if existing and existing.get('status') == 'OK':
            sections = existing.get('sections', {})
            passages = existing.get('passages', [])
            status.update(
                {
                    'corp_code': existing.get('corp_code', ''),
                    'report_nm': existing.get('report_nm', ''),
                    'rcept_no': existing.get('rcept_no', ''),
                    'rcept_dt': existing.get('rcept_dt', ''),
                    'viewer_url': existing.get('viewer_url', ''),
                    'status': 'SKIP_EXISTING',
                    'section_II_chars': len(sections.get('II') or ''),
                    'section_IV_chars': len(sections.get('IV') or ''),
                    'section_VI_chars': len(sections.get('VI') or ''),
                    'passage_count': len(passages),
                }
            )
            passage_rows = [
                {
                    'company_name': company_name,
                    'stock_code': stock_code,
                    'corp_code': existing.get('corp_code', ''),
                    'esg_year': esg_year,
                    'report_fiscal_year': fiscal_year,
                    'rcept_no': existing.get('rcept_no', ''),
                    'rcept_dt': existing.get('rcept_dt', ''),
                    'viewer_url': existing.get('viewer_url', ''),
                    **passage,
                }
                for passage in passages
            ]
            return status, passage_rows

    corp_code = stock_to_corp.get(stock_code)
    if not corp_code:
        status.update({'failure_stage': 'corp_code', 'failure_reason': 'stock_code에 대응하는 corp_code 없음'})
        return status, []
    status['corp_code'] = corp_code

    try:
        selection = select_annual_report(corp_code, stock_code, fiscal_year)
    except Exception as exc:
        status.update({'failure_stage': 'list.json', 'failure_reason': str(exc)})
        return status, []

    if selection is None:
        status.update({'failure_stage': 'rcept_no', 'failure_reason': '대상 회계연도 사업보고서 없음'})
        return status, []

    status.update(
        {
            'report_nm': selection.report_nm,
            'rcept_no': selection.rcept_no,
            'rcept_dt': selection.rcept_dt,
            'viewer_url': selection.viewer_url,
        }
    )
    time.sleep(SLEEP_BETWEEN_CALLS)

    try:
        xml_files = fetch_report_xml_files(selection.rcept_no)
        sections = extract_target_sections(xml_files)
    except Exception as exc:
        status.update({'failure_stage': 'document.xml', 'failure_reason': str(exc)})
        return status, []

    missing_sections = [label for label in TARGET_SECTION_LABELS if not sections.get(label)]
    if missing_sections:
        status.update(
            {
                'failure_stage': 'section_extract',
                'failure_reason': f'섹션 추출 실패: {missing_sections}',
                'section_II_chars': len(sections.get('II') or ''),
                'section_IV_chars': len(sections.get('IV') or ''),
                'section_VI_chars': len(sections.get('VI') or ''),
            }
        )
        return status, []

    passages = extract_esg_passages(sections, seed_patterns)
    status.update(
        {
            'status': 'OK',
            'section_II_chars': len(sections.get('II') or ''),
            'section_IV_chars': len(sections.get('IV') or ''),
            'section_VI_chars': len(sections.get('VI') or ''),
            'passage_count': len(passages),
        }
    )

    json_payload = {
        'status': 'OK',
        'company_name': company_name,
        'stock_code': stock_code,
        'corp_code': corp_code,
        'fiscal_year': fiscal_year,
        'esg_year': esg_year,
        'report_fiscal_year': fiscal_year,
        'report_nm': selection.report_nm,
        'rcept_no': selection.rcept_no,
        'rcept_dt': selection.rcept_dt,
        'viewer_url': selection.viewer_url,
        'sections': sections,
        'passages': passages,
    }
    write_json_atomic(output_path, json_payload)

    passage_rows = [
        {
            'company_name': company_name,
            'stock_code': stock_code,
            'corp_code': corp_code,
            'esg_year': esg_year,
            'report_fiscal_year': fiscal_year,
            'rcept_no': selection.rcept_no,
            'rcept_dt': selection.rcept_dt,
            'viewer_url': selection.viewer_url,
            **passage,
        }
        for passage in passages
    ]
    return status, passage_rows

## 10. 1-2개 기업 테스트 실행

전체 381행을 실행하기 전에 삼성전자 같은 단건으로 API 키, 보고서 선택, 섹션 추출, passage 추출이 모두 작동하는지 확인합니다.

In [23]:
def run_sample(stock_code: str = '005930', fiscal_year: int = 2024) -> tuple[pd.DataFrame, pd.DataFrame]:
    """단건 테스트를 실행합니다."""
    target = company_master[
        (company_master['stock_code'] == stock_code.zfill(6))
        & (company_master['fiscal_year'] == fiscal_year)
    ]
    if target.empty:
        raise ValueError(f'company_master에 {stock_code}, {fiscal_year} 행이 없습니다.')

    row_index = int(target.index[0]) + 1
    status, passages = process_firm_year(row_index, target.iloc[0], skip_existing=False)
    status_df = pd.DataFrame([status], columns=STATUS_COLUMNS)
    passages_df = pd.DataFrame(passages, columns=PASSAGE_COLUMNS)

    print(status_df.T)
    print(f'passage 후보 수: {len(passages_df)}')
    return status_df, passages_df


# 네트워크 호출이 발생합니다. 준비되면 주석을 해제해 실행하세요.
sample_status, sample_passages = run_sample('005930', 2024)
sample_passages.head()

                                                                  0
row_index                                                         3
company_name                                                   삼성전자
stock_code                                                   005930
corp_code                                                  00126380
fiscal_year                                                    2024
esg_year                                                       2025
report_nm                                           사업보고서 (2024.12)
rcept_no                                             20250311001085
rcept_dt                                                   20250311
viewer_url        https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
status                                                           OK
failure_stage                                                      
failure_reason                                                     
section_II_chars                                

,company_name,stock_code,corp_code,esg_year,report_fiscal_year,rcept_no,rcept_dt,viewer_url,section,passage,dimension,matched_terms
0,삼성전자,005930,00126380,2025,2024,20250311001085,20250311,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,II,당사의 자본관리 목적은 계속기업으로서 주주 및 이해관계자에게 이익을 지속적으로제공할...,G,주주
1,삼성전자,005930,00126380,2025,2024,20250311001085,20250311,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,II,"그리고 당사는 레인보우로보틱스㈜ 및 그 최대주주 등과 체결한 주주간 계약에 따라,일...",G,주주
2,삼성전자,005930,00126380,2025,2024,20250311001085,20250311,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,II,"또한 레인보우로보틱스㈜의 최대주주 등은 일정한 사유가 발생할 경우, 당사가 보유한 ...",G,주주
3,삼성전자,005930,00126380,2025,2024,20250311001085,20250311,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,II,당사는 2024년 12월 31일 이사회에서 레인보우로보틱스㈜의 최대주주 등 6인이 ...,G,이사회|주주
4,삼성전자,005930,00126380,2025,2024,20250311001085,20250311,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,II,"2024년말 현재 해당 콜옵션의 공정가치는 안진회계법인이 평가하였습니다.또한, 종속...",G,주주


## 11. 전체 381행 배치 실행

아래 셀을 실행하면 전체 입력 행을 순서대로 처리합니다. 성공, 실패, 기존 파일 재사용 모두 `dart_collection_status.csv`에 기록합니다.

In [24]:
def collect_all(
    df: pd.DataFrame = company_master,
    start: int = 1,
    end: int | None = None,
    skip_existing: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """company_master 구간을 처리하고 CSV 산출물을 저장합니다."""
    if end is None:
        end = len(df)
    target = df.iloc[start - 1:end]

    all_status: list[dict[str, Any]] = []
    all_passages: list[dict[str, Any]] = []

    print(f'수집 시작: {start}-{end}행, 총 {len(target)}건')
    started_at = time.time()

    for offset, (_, row) in enumerate(target.iterrows(), start=0):
        row_index = start + offset
        label = f'{row_index:03d}/{end:03d} {row["stock_code"]} {int(row["fiscal_year"])} {row.get("company_name", "")}'
        try:
            status, passages = process_firm_year(row_index, row, skip_existing=skip_existing)
        except Exception as exc:
            status = make_empty_status(row_index, row)
            status.update({'failure_stage': 'unhandled', 'failure_reason': str(exc)})
            passages = []

        all_status.append(status)
        all_passages.extend(passages)
        print(f'{label}: {status["status"]}, passages={status["passage_count"]}, reason={status["failure_reason"]}')
        time.sleep(SLEEP_BETWEEN_CALLS)

    status_df = pd.DataFrame(all_status, columns=STATUS_COLUMNS)
    passages_df = pd.DataFrame(all_passages, columns=PASSAGE_COLUMNS)

    status_df.to_csv(STATUS_CSV_PATH, index=False, encoding='utf-8-sig')
    passages_df.to_csv(PASSAGES_CSV_PATH, index=False, encoding='utf-8-sig')

    elapsed = time.time() - started_at
    print(f'완료: {elapsed:.1f}초')
    print(status_df['status'].value_counts(dropna=False))
    print(f'저장: {STATUS_CSV_PATH}')
    print(f'저장: {PASSAGES_CSV_PATH}')
    return status_df, passages_df


# 전체 실행입니다. OpenDART API 호출이 많이 발생합니다.
status_df, passages_df = collect_all(skip_existing=True)

수집 시작: 1-381행, 총 381건
001/381 005930 2022 삼성전자: SKIP_EXISTING, passages=142, reason=
002/381 005930 2023 삼성전자: SKIP_EXISTING, passages=125, reason=
003/381 005930 2024 삼성전자: SKIP_EXISTING, passages=136, reason=
004/381 001460 2022 BYC: SKIP_EXISTING, passages=43, reason=
005/381 001460 2023 BYC: SKIP_EXISTING, passages=47, reason=
006/381 001460 2024 BYC: SKIP_EXISTING, passages=63, reason=
007/381 005380 2022 현대자동차: SKIP_EXISTING, passages=183, reason=
008/381 005380 2023 현대자동차: SKIP_EXISTING, passages=198, reason=
009/381 005380 2024 현대자동차: SKIP_EXISTING, passages=225, reason=
010/381 000120 2022 CJ대한통운: SKIP_EXISTING, passages=94, reason=
011/381 000120 2023 CJ대한통운: SKIP_EXISTING, passages=107, reason=
012/381 000120 2024 CJ대한통운: SKIP_EXISTING, passages=105, reason=
013/381 097950 2022 CJ제일제당: SKIP_EXISTING, passages=116, reason=
014/381 097950 2023 CJ제일제당: SKIP_EXISTING, passages=120, reason=
015/381 097950 2024 CJ제일제당: SKIP_EXISTING, passages=115, reason=
016/381 001040 2022 CJ: S

KeyboardInterrupt: 

## 12. 수집 결과 검증 및 실패 사유 요약

수집 후에는 성공 행의 식별자와 섹션 길이를 확인하고, 실패 행은 최종 보고서에 설명할 수 있도록 단계별로 요약합니다.

In [ ]:
def validate_collection_outputs(
    status_path: Path = STATUS_CSV_PATH,
    passages_path: Path = PASSAGES_CSV_PATH,
) -> dict[str, Any]:
    """저장된 수집 산출물을 점검합니다."""
    if not status_path.exists():
        raise FileNotFoundError(f'상태 CSV가 없습니다: {status_path}')

    status_df = pd.read_csv(status_path, dtype={'stock_code': 'string', 'corp_code': 'string', 'rcept_no': 'string'})
    passages_df = pd.read_csv(passages_path, dtype={'stock_code': 'string', 'corp_code': 'string', 'rcept_no': 'string'}) if passages_path.exists() else pd.DataFrame(columns=PASSAGE_COLUMNS)

    ok_mask = status_df['status'].isin(['OK', 'SKIP_EXISTING'])
    ok_df = status_df[ok_mask]

    issues = []
    if len(status_df) != 381:
        issues.append(f'status CSV 행 수가 381이 아닙니다: {len(status_df)}')
    for col in ['rcept_no', 'viewer_url']:
        missing = int(ok_df[col].isna().sum() + (ok_df[col].fillna('') == '').sum())
        if missing:
            issues.append(f'성공 행 중 {col} 누락: {missing}')
    for label in TARGET_SECTION_LABELS:
        col = f'section_{label}_chars'
        short_count = int((ok_df[col].fillna(0) <= 0).sum())
        if short_count:
            issues.append(f'성공 행 중 {label} 섹션 길이 0: {short_count}')

    failure_summary = (
        status_df[~ok_mask]
        .groupby(['failure_stage', 'failure_reason'], dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )

    summary = {
        'status_rows': len(status_df),
        'passage_rows': len(passages_df),
        'success_rows': int(ok_mask.sum()),
        'failure_rows': int((~ok_mask).sum()),
        'issues': issues,
    }

    print(json.dumps(summary, ensure_ascii=False, indent=2))
    if len(failure_summary):
        display(failure_summary.head(30))
    return summary


# 수집 후 실행하세요.
output_validation = validate_collection_outputs()

{
  "status_rows": 381,
  "passage_rows": 39322,
  "success_rows": 376,
  "failure_rows": 5,
  "issues": []
}


,failure_stage,failure_reason,count
0,rcept_no,대상 회계연도 사업보고서 없음,5


## 13. 비-12월 결산 회사 재수집

### 실패 5건 원인 분석

`failure_stage='rcept_no'`로 떨어진 5행은 모두 **12월 결산이 아닌 회사**입니다.

| row_index | stock_code | 회사 | fiscal_year | 결산월 | KCGS 등급 |
|---|---|---|---|---|---|
| 223 | 001720 | 신영증권 | 2022 | 3월 | B+ |
| 224 | 001720 | 신영증권 | 2023 | 3월 | B+ |
| 225 | 001720 | 신영증권 | 2024 | 3월 | B+ |
| 352 | 001080 | 만호제강 | 2022 | 6월 | C |
| 353 | 001080 | 만호제강 | 2023 | 6월 | D |

ESG 등급은 5건 모두 한국ESG기준원에서 정상 책정되어 있어 분석 대상에서 제외할 수 없는 케이스입니다.

**근본 원인.** `select_annual_report`의 `report_name_has_target_year`는 보고서명에 `(fiscal_year.)` 패턴이 있는지로 회계연도를 식별합니다. 그런데 DART 보고서명의 `(YYYY.MM)`은 **회계연도 종료시점**을 의미하므로 결산월이 12월이 아닌 회사는 다음과 같이 어긋납니다.

| 결산월 | fiscal_year=2022 회계기간 | 실제 보고서명 | 기존 매칭 결과 |
|---|---|---|---|
| 12월 (대다수) | 2022.01–2022.12 | `사업보고서 (2022.12)` | OK |
| 3월 (신영증권) | 2022.04–2023.03 | `사업보고서 (2023.03)` | 미매칭 |
| 6월 (만호제강) | 2022.07–2023.06 | `사업보고서 (2023.06)` | 미매칭 |

**참고 — 만호제강 2024 캐시 정합성 의심.** 로그에서 `001080 2024`가 `SKIP_EXISTING, passages=71`로 통과된 이유는, 과거 수집 시 disclosure_year=2025의 `[기재정정]사업보고서 (2024.06)`(회계기간 2023.07–2024.06)이 `\(2024\.` 패턴에 우연히 매칭되어 캐시된 결과입니다. esg_year=2025 평가 기준일과 한 회계연도 일찍 매핑된 셈이라, 새 매핑 규칙으로 보면 fiscal_year=2024는 `(2025.06)`이 정답입니다. 아래 재수집 셀에서 같은 규칙으로 갱신합니다.

### 재수집 전략

1. `corp.json`에서 `acc_mt`(결산월)을 조회합니다.
2. 12월 결산이면 기존과 동일하게 `(fiscal_year.12)`를 찾습니다.
3. 비-12월 결산이면 `(fiscal_year+1.MM)`을 찾습니다. 예: 신영증권 fiscal_year=2022 → `(2023.03)`.
4. 정정본까지 포함하기 위해 공시연도 윈도우를 `disclosure_year` ~ `disclosure_year+1`로 확장합니다.
5. 이후 처리(document.xml → 섹션 추출 → passage 추출 → JSON 저장)는 기존 함수를 재사용합니다.
6. 결과를 `dart_collection_status.csv`와 `dart_passages.csv`에 머지해 한 번에 일관된 상태로 만듭니다.


In [27]:
FAILED_ROWS: list[tuple[str, int]] = [
    ('001720', 2022),  # 신영증권 (3월 결산)
    ('001720', 2023),
    ('001720', 2024),
    ('001080', 2022),  # 만호제강 (6월 결산)
    ('001080', 2023),
    ('001080', 2024),  # 기존 SKIP_EXISTING 캐시 정합성 갱신
]


def fetch_acc_mt(corp_code: str) -> int:
    """OpenDART company.json에서 결산월(acc_mt)을 조회합니다."""
    payload = opendart_get_json('company.json', {'corp_code': corp_code})
    if payload.get('status') != '000':
        raise RuntimeError(f'company.json 에러: {payload.get("status")} {payload.get("message")}')
    raw = (payload.get('acc_mt') or '12').strip()
    return int(raw) if raw else 12


def fiscal_year_label(fiscal_year: int, acc_mt: int) -> tuple[int, int]:
    """결산월을 반영한 (보고서명 연도, 보고서명 월)을 돌려줍니다."""
    if acc_mt == 12:
        return fiscal_year, 12
    return fiscal_year + 1, acc_mt


def report_priority_fiscal_aware(row: dict[str, Any]) -> tuple[int, int]:
    """결산월 매칭이 끝난 뒤 정정본/원본 우선순위를 매깁니다."""
    name = row.get('report_nm', '')
    if '[첨부정정]' in name:
        correction_score = 8
    elif '[기재정정]' in name:
        correction_score = 0
    elif name.replace(' ', '').startswith('사업보고서'):
        correction_score = 1
    else:
        correction_score = 4
    return (correction_score, -int(row.get('rcept_no', '0') or '0'))


def select_annual_report_fiscal_aware(
    corp_code: str,
    stock_code: str,
    fiscal_year: int,
    acc_mt: int,
) -> ReportSelection | None:
    """결산월을 반영해 사업보고서 후보를 선택합니다.

    공시연도는 `fiscal_year + 1`부터 `fiscal_year + 2`까지 훑어
    정정본까지 후보에 포함시킵니다.
    """
    target_year, target_month = fiscal_year_label(fiscal_year, acc_mt)
    disclosure_start = fiscal_year + 1
    disclosure_end = fiscal_year + 2

    candidates: list[dict[str, Any]] = []
    for disclosure_year in range(disclosure_start, disclosure_end + 1):
        try:
            reports = list_annual_reports(corp_code, disclosure_year - 1)
        except RuntimeError:
            continue
        candidates.extend(reports)

    if not candidates:
        return None

    label_pattern = re.compile(rf'\({target_year}\.0?{target_month}[.)]')
    matched = [
        row for row in candidates
        if label_pattern.search(re.sub(r'\s+', '', row.get('report_nm', '')))
    ]
    if not matched:
        return None

    selected = sorted(matched, key=report_priority_fiscal_aware)[0]
    return ReportSelection(
        report_nm=selected.get('report_nm', ''),
        rcept_no=selected.get('rcept_no', ''),
        rcept_dt=selected.get('rcept_dt', ''),
        corp_code=corp_code,
        stock_code=stock_code,
        fiscal_year=fiscal_year,
    )


def process_failed_firm_year(stock_code: str, fiscal_year: int) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    """결산월 인식 매칭으로 5개 실패 행을 다시 처리합니다."""
    target = company_master[
        (company_master['stock_code'] == stock_code)
        & (company_master['fiscal_year'] == fiscal_year)
    ]
    if target.empty:
        raise ValueError(f'company_master에 {stock_code} {fiscal_year} 행이 없습니다.')

    row = target.iloc[0]
    row_index = int(target.index[0]) + 1
    status = make_empty_status(row_index, row)
    company_name = row.get('company_name', '')
    esg_year = int(row.get('esg_year'))

    corp_code = stock_to_corp.get(stock_code)
    if not corp_code:
        status.update({'failure_stage': 'corp_code', 'failure_reason': 'stock_code에 대응하는 corp_code 없음'})
        return status, []
    status['corp_code'] = corp_code

    try:
        acc_mt = fetch_acc_mt(corp_code)
    except Exception as exc:
        status.update({'failure_stage': 'company.json', 'failure_reason': str(exc)})
        return status, []
    time.sleep(SLEEP_BETWEEN_CALLS)

    try:
        selection = select_annual_report_fiscal_aware(corp_code, stock_code, fiscal_year, acc_mt)
    except Exception as exc:
        status.update({'failure_stage': 'list.json', 'failure_reason': str(exc)})
        return status, []

    if selection is None:
        target_year, target_month = fiscal_year_label(fiscal_year, acc_mt)
        status.update({
            'failure_stage': 'rcept_no',
            'failure_reason': f'결산월 {acc_mt} 기준 ({target_year}.{target_month:02d}) 보고서 미발견',
        })
        return status, []

    status.update({
        'report_nm': selection.report_nm,
        'rcept_no': selection.rcept_no,
        'rcept_dt': selection.rcept_dt,
        'viewer_url': selection.viewer_url,
    })
    time.sleep(SLEEP_BETWEEN_CALLS)

    try:
        xml_files = fetch_report_xml_files(selection.rcept_no)
        sections = extract_target_sections(xml_files)
    except Exception as exc:
        status.update({'failure_stage': 'document.xml', 'failure_reason': str(exc)})
        return status, []

    missing = [label for label in TARGET_SECTION_LABELS if not sections.get(label)]
    if missing:
        status.update({
            'failure_stage': 'section_extract',
            'failure_reason': f'섹션 추출 실패: {missing}',
            'section_II_chars': len(sections.get('II') or ''),
            'section_IV_chars': len(sections.get('IV') or ''),
            'section_VI_chars': len(sections.get('VI') or ''),
        })
        return status, []

    passages = extract_esg_passages(sections, seed_patterns)
    status.update({
        'status': 'OK',
        'section_II_chars': len(sections.get('II') or ''),
        'section_IV_chars': len(sections.get('IV') or ''),
        'section_VI_chars': len(sections.get('VI') or ''),
        'passage_count': len(passages),
    })

    json_payload = {
        'status': 'OK',
        'company_name': company_name,
        'stock_code': stock_code,
        'corp_code': corp_code,
        'fiscal_year': fiscal_year,
        'esg_year': esg_year,
        'report_fiscal_year': fiscal_year,
        'acc_mt': acc_mt,
        'report_nm': selection.report_nm,
        'rcept_no': selection.rcept_no,
        'rcept_dt': selection.rcept_dt,
        'viewer_url': selection.viewer_url,
        'sections': sections,
        'passages': passages,
    }
    write_json_atomic(report_json_path(stock_code, fiscal_year), json_payload)

    passage_rows = [
        {
            'company_name': company_name,
            'stock_code': stock_code,
            'corp_code': corp_code,
            'esg_year': esg_year,
            'report_fiscal_year': fiscal_year,
            'rcept_no': selection.rcept_no,
            'rcept_dt': selection.rcept_dt,
            'viewer_url': selection.viewer_url,
            **passage,
        }
        for passage in passages
    ]
    return status, passage_rows


def merge_recollection_into_outputs(
    new_status_rows: list[dict[str, Any]],
    new_passage_rows: list[dict[str, Any]],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """5건 재수집 결과를 status/passages CSV에 반영해 저장합니다.

    기존 CSV는 일부 컬럼이 NaN 때문에 float64로 읽혀 string 대입 시
    dtype 충돌이 납니다. 행 단위로 삭제→재삽입하여 dtype 문제를 피합니다.
    """
    status_df = pd.read_csv(
        STATUS_CSV_PATH,
        dtype={'stock_code': 'string', 'corp_code': 'string', 'rcept_no': 'string'},
    )
    if PASSAGES_CSV_PATH.exists():
        passages_df = pd.read_csv(
            PASSAGES_CSV_PATH,
            dtype={'stock_code': 'string', 'corp_code': 'string', 'rcept_no': 'string'},
        )
    else:
        passages_df = pd.DataFrame(columns=PASSAGE_COLUMNS)

    touched_keys = {(row['stock_code'], int(row['fiscal_year'])) for row in new_status_rows}

    if touched_keys and len(status_df):
        drop_mask = status_df.apply(
            lambda r: (str(r['stock_code']).zfill(6), int(r['fiscal_year'])) in touched_keys,
            axis=1,
        )
        status_df = status_df[~drop_mask].reset_index(drop=True)

    status_df = pd.concat(
        [status_df, pd.DataFrame(new_status_rows, columns=STATUS_COLUMNS)],
        ignore_index=True,
    )
    status_df['row_index'] = status_df['row_index'].astype(int)
    status_df = status_df.sort_values('row_index').reset_index(drop=True)

    if touched_keys and len(passages_df):
        drop_mask = passages_df.apply(
            lambda r: (str(r['stock_code']).zfill(6), int(r['report_fiscal_year'])) in touched_keys,
            axis=1,
        )
        passages_df = passages_df[~drop_mask].reset_index(drop=True)

    if new_passage_rows:
        passages_df = pd.concat(
            [passages_df, pd.DataFrame(new_passage_rows, columns=PASSAGE_COLUMNS)],
            ignore_index=True,
        )

    status_df.to_csv(STATUS_CSV_PATH, index=False, encoding='utf-8-sig')
    passages_df.to_csv(PASSAGES_CSV_PATH, index=False, encoding='utf-8-sig')
    return status_df, passages_df


new_statuses: list[dict[str, Any]] = []
new_passages: list[dict[str, Any]] = []

for stock_code, fiscal_year in FAILED_ROWS:
    print(f'--- {stock_code} {fiscal_year} ---')
    status, passages = process_failed_firm_year(stock_code, fiscal_year)
    print(
        f'   status={status["status"]} '
        f'report_nm={status["report_nm"]!r} '
        f'rcept_no={status["rcept_no"]} '
        f'passages={status["passage_count"]} '
        f'reason={status["failure_reason"]}'
    )
    new_statuses.append(status)
    new_passages.extend(passages)
    time.sleep(SLEEP_BETWEEN_CALLS)

merged_status, merged_passages = merge_recollection_into_outputs(new_statuses, new_passages)

print('\n=== 갱신 후 상태 분포 ===')
print(merged_status['status'].value_counts(dropna=False))
print(f'전체 status 행: {len(merged_status):,}')
print(f'전체 passage 행: {len(merged_passages):,}')
print(f'저장: {STATUS_CSV_PATH}')
print(f'저장: {PASSAGES_CSV_PATH}')

merged_status[merged_status['stock_code'].isin(['001720', '001080'])][
    ['stock_code', 'company_name', 'fiscal_year', 'status', 'report_nm', 'rcept_no', 'passage_count', 'failure_reason']
]

--- 001720 2022 ---
   status=OK report_nm='사업보고서 (2023.03)' rcept_no=20230615000542 passages=52 reason=
--- 001720 2023 ---
   status=OK report_nm='사업보고서 (2024.03)' rcept_no=20240612000337 passages=65 reason=
--- 001720 2024 ---
   status=OK report_nm='[기재정정]사업보고서 (2025.03)' rcept_no=20250829000681 passages=64 reason=
--- 001080 2022 ---
   status=OK report_nm='사업보고서 (2023.06)' rcept_no=20231005000147 passages=29 reason=
--- 001080 2023 ---
   status=OK report_nm='[기재정정]사업보고서 (2024.06)' rcept_no=20250930000631 passages=71 reason=
--- 001080 2024 ---
   status=OK report_nm='[기재정정]사업보고서 (2025.06)' rcept_no=20250918000340 passages=58 reason=

=== 갱신 후 상태 분포 ===
status
SKIP_EXISTING    375
OK                 6
Name: count, dtype: int64
전체 status 행: 381
전체 passage 행: 39,590
저장: /Users/home1/Documents/esg-dart-project/data/processed/dart_collection_status.csv
저장: /Users/home1/Documents/esg-dart-project/data/processed/dart_passages.csv


,stock_code,company_name,fiscal_year,status,report_nm,rcept_no,passage_count,failure_reason
222,001720,신영증권,2022,OK,사업보고서 (2023.03),20230615000542,52,
223,001720,신영증권,2023,OK,사업보고서 (2024.03),20240612000337,65,
224,001720,신영증권,2024,OK,[기재정정]사업보고서 (2025.03),20250829000681,64,
351,001080,만호제강,2022,OK,사업보고서 (2023.06),20231005000147,29,
352,001080,만호제강,2023,OK,[기재정정]사업보고서 (2024.06),20250930000631,71,
353,001080,만호제강,2024,OK,[기재정정]사업보고서 (2025.06),20250918000340,58,
